# Lecture 5: Machine Learning Fundamentals
**BITE 485 | Tharaka University**

---

## Learning Outcomes
1. Implement a supervised learning pipeline end-to-end
2. Diagnose overfitting and underfitting
3. Apply cross-validation
4. Apply regularisation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
import warnings; warnings.filterwarnings('ignore')
print('Ready.')

## 5.1 Loading the Titanic Dataset

In [ ]:
try:
    titanic = fetch_openml('titanic', version=1, as_frame=True, parser='auto')
    df = titanic.frame[['pclass','age','sibsp','parch','fare','survived']].copy()
    df.columns = ['pclass','age','sibsp','parch','fare','survived']
    df = df.dropna()
    df = df.astype(float)
    print(f'Titanic dataset loaded: {df.shape}')
except Exception:
    print('Generating synthetic dataset...')
    np.random.seed(42)
    n = 600
    df = pd.DataFrame({
        'pclass':   np.random.choice([1,2,3], n, p=[0.2,0.3,0.5]),
        'age':      np.random.normal(30, 14, n).clip(1, 80),
        'sibsp':    np.random.poisson(0.5, n),
        'parch':    np.random.poisson(0.4, n),
        'fare':     np.random.exponential(30, n),
        'survived': np.random.choice([0,1], n, p=[0.62,0.38])
    })
print(df.head())
print(f'\nSurvival rate: {df.survived.mean()*100:.1f}%')

In [ ]:
X = df.drop(columns='survived')
y = df['survived']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 5.2 Demonstrating Overfitting

In [ ]:
train_acc, test_acc = [], []
depths = range(1, 21)
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, dt.predict(X_train)))
    test_acc.append(accuracy_score(y_test,  dt.predict(X_test)))

plt.figure(figsize=(9, 5))
plt.plot(depths, train_acc, 'b-o', label='Training Accuracy')
plt.plot(depths, test_acc,  'r-o', label='Test Accuracy')
plt.xlabel('Max Tree Depth')
plt.ylabel('Accuracy')
plt.title('Bias-Variance Trade-off: Overfitting as Depth Increases')
plt.legend()
plt.axvline(x=test_acc.index(max(test_acc))+1, color='green',
            linestyle='--', label=f'Optimal depth={test_acc.index(max(test_acc))+1}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Best test accuracy at depth {test_acc.index(max(test_acc))+1}: {max(test_acc):.4f}')

## 5.3 Cross-Validation

In [ ]:
model = DecisionTreeClassifier(max_depth=5, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
print('=== 5-Fold Cross-Validation Results ===')
for i, s in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {s:.4f}')
print(f'\nMean CV Accuracy:  {cv_scores.mean():.4f}')
print(f'CV Std Deviation:  {cv_scores.std():.4f}')
print(f'\nFinal Test Accuracy: {accuracy_score(y_test, model.fit(X_train,y_train).predict(X_test)):.4f}')

## 5.4 Regularisation with Logistic Regression

In [ ]:
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

results = []
for C in [0.001, 0.01, 0.1, 1, 10, 100]:
    lr = LogisticRegression(C=C, random_state=42, max_iter=1000)
    lr.fit(X_tr_sc, y_train)
    results.append({'C (1/lambda)': C,
                    'Train Acc': accuracy_score(y_train, lr.predict(X_tr_sc)),
                    'Test Acc':  accuracy_score(y_test,  lr.predict(X_te_sc))})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
print('\nNote: C controls regularisation strength. Smaller C = stronger regularisation.')

### Exercise

Using the Titanic dataset:

1. Build a complete ML pipeline using `sklearn.pipeline.Pipeline` that:
   - Scales features with StandardScaler
   - Classifies with LogisticRegression(C=1.0)
2. Evaluate with 5-fold cross-validation.
3. Compare the pipeline's mean CV accuracy with a DecisionTreeClassifier(max_depth=5).
4. Which model would you recommend for deployment and why?

In [ ]:
# Your code here


---
*BITE 485 Data Science | Tharaka University | kevin.tuei@tharaka.ac.ke*